# Conservative 2D regrid — curvilinear target

Curvilinear grids have 2D `lat(y, x)` / `lon(y, x)` coordinate arrays —
ocean models (ORCA, tripolar), rotated regional forecasts. Not
1D-separable, so `.conservative` can't handle them;
`.regrid.conservative_2d` is the tool.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder

## Source — regular 1° lat/lon, analytic two-bump field

In [ ]:
lat = np.linspace(-60, 60, 121)
lon = np.linspace(-120, 120, 241)
Lo, La = np.meshgrid(lon, lat)
field = (
    np.exp(-((Lo - 40) ** 2 + (La - 20) ** 2) / 500)
    - np.exp(-((Lo + 60) ** 2 + (La + 15) ** 2) / 400)
)
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat, "longitude": lon},
)
src.plot(figsize=(8, 3.5), cmap="RdBu_r", center=0)
plt.title("source: analytic two-bump field")
plt.tight_layout()

## Target — rotated curvilinear grid

Coordinates ride on a `(ny, nx)` mesh, stored as 2D coordinate
variables on the target Dataset.

In [ ]:
ny, nx = 30, 50
xi, yi = np.meshgrid(
    np.linspace(-110, 110, nx),
    np.linspace(-45, 45, ny),
    indexing="xy",
)
th = np.deg2rad(30)
lon2d = xi * np.cos(th) - yi * np.sin(th)
lat2d = xi * np.sin(th) + yi * np.cos(th)
target = xr.Dataset(coords={
    "longitude": (("ny", "nx"), lon2d),
    "latitude":  (("ny", "nx"), lat2d),
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lon2d, lat2d, color="0.3", lw=0.4)
ax.plot(lon2d.T, lat2d.T, color="0.3", lw=0.4)
ax.set_title("curvilinear target (30° rotation)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Regrid and plot

For the one-shot accessor, use `src.regrid.conservative_2d(target, ...)`.
Constructing the class directly (as below) is equivalent but lets us reuse
the weight matrix for both the apply and the diagnostic in the next cell.

In [ ]:
rgr = ConservativeRegridder(
    src, target, x_coord="longitude", y_coord="latitude",
)
regridded = rgr.regrid(src)

fig, ax = plt.subplots(figsize=(8, 4))
pc = ax.pcolormesh(lon2d, lat2d, regridded.values, cmap="RdBu_r",
                   shading="auto", vmin=-1, vmax=1)
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title("regridded onto rotated curvilinear grid")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Conservation check

For the target cells that fall within the source domain, the
area-weighted sum of outputs equals the direct A·s integral to
machine precision.

In [ ]:
A = rgr._areas
src_cover = np.ravel(A.sum(axis=0).todense())
tgt_cover = A.sum(axis=1).todense().reshape(regridded.shape)
valid = np.isfinite(regridded.values)

direct = float((src.values.ravel() * src_cover).sum())
via_regrid = float((regridded.values[valid] * tgt_cover[valid]).sum())
print(f"direct        : {direct:.6f}")
print(f"via regrid    : {via_regrid:.6f}")
print(f"relative err  : {abs(direct - via_regrid) / max(abs(direct), 1e-12):.2e}")
print(f"coverage      : {valid.mean():.2%} of target cells inside source")